# Hyperspherical VAE for Semantic Clustering Validation

이 노트북은 von Mises-Fisher (vMF) distribution을 사용하여 latent space를 hypersphere에 위치시키고, semantic cluster가 유지되는지 확인합니다.

**목표:**
- vMF-VAE 구현 (hypersphere constrained latent)
- semantic_wm 데이터셋 사용 (normal, violence, sexual)
- Semantic cluster 유지 확인
- t-SNE 시각화 및 cluster metrics 평가

## 1. Setup and Imports

## 0. Google Drive Mount (Colab)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
import os
if os.path.exists('/content/drive/MyDrive'):
    print("✓ Google Drive mounted successfully!")
else:
    print("✗ Mount failed. Please check the path.")

In [ ]:
# ============================================================================
# Google Drive: dataset extraction (skip this cell when running locally)
# Expects /content/drive/MyDrive/semantic_wm/dataset.zip — see README.
# ============================================================================
zip_path = '/content/drive/MyDrive/semantic_wm/dataset.zip'

print("Unzipping dataset...")
!unzip -q {zip_path} -d /content/semantic_wm/
print("Done.")

import os
if os.path.exists('/content/semantic_wm/dataset'):
    print("✓ Dataset extracted to /content/semantic_wm/dataset")
else:
    print("⚠ Dataset not found. Check that semantic_wm/dataset.zip is on Drive.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from PIL import Image
import os
from tqdm import tqdm
from collections import defaultdict
import scipy.special
from numbers import Number
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. von Mises-Fisher Distribution Implementation

GitHub repo (nicola-decao/s-vae-pytorch)를 참고하여 구현합니다.

In [ ]:
# Modified Bessel function of the first kind (exponentially scaled)
class IveFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, v, z):
        assert isinstance(v, Number), "v must be a scalar"

        ctx.save_for_backward(z)
        ctx.v = v
        z_cpu = z.data.cpu().numpy()

        if np.isclose(v, 0):
            output = scipy.special.i0e(z_cpu, dtype=z_cpu.dtype)
        elif np.isclose(v, 1):
            output = scipy.special.i1e(z_cpu, dtype=z_cpu.dtype)
        else:
            output = scipy.special.ive(v, z_cpu, dtype=z_cpu.dtype)

        return torch.Tensor(output).to(z.device)

    @staticmethod
    def backward(ctx, grad_output):
        z = ctx.saved_tensors[-1]
        return (
            None,
            grad_output * (ive(ctx.v - 1, z) - ive(ctx.v, z) * (ctx.v + z) / z),
        )

ive = IveFunction.apply

In [ ]:
# Hyperspherical Uniform Distribution
class HypersphericalUniform(torch.distributions.Distribution):
    support = torch.distributions.constraints.real
    has_rsample = False
    _mean_carrier_measure = 0

    def __init__(self, dim, validate_args=None, device="cpu"):
        super(HypersphericalUniform, self).__init__(
            torch.Size([dim]), validate_args=validate_args
        )
        self._dim = dim
        self._device = device if isinstance(device, torch.device) else torch.device(device)

    @property
    def dim(self):
        return self._dim

    @property
    def device(self):
        return self._device

    def sample(self, shape=torch.Size()):
        output = (
            torch.distributions.Normal(0, 1)
            .sample(
                (shape if isinstance(shape, torch.Size) else torch.Size([shape]))
                + torch.Size([self._dim + 1])
            )
            .to(self.device)
        )
        return output / output.norm(dim=-1, keepdim=True)

    def entropy(self):
        return self.__log_surface_area()

    def log_prob(self, x):
        return -torch.ones(x.shape[:-1], device=self.device) * self.__log_surface_area()

    def __log_surface_area(self):
        lgamma = torch.lgamma(torch.tensor([(self._dim + 1) / 2]).to(self.device))
        return math.log(2) + ((self._dim + 1) / 2) * math.log(math.pi) - lgamma

In [ ]:
# Bessel function ratio approximations (from papers)
# Source: https://arxiv.org/pdf/1606.02008.pdf
def ive_fraction_approx(v, z):
    """
    Approximation of I_v(z) / I_{v-1}(z)
    This is a lower bound that is numerically stable.
    """
    return z / (v - 1 + torch.sqrt(torch.pow(v + 1, 2) + torch.pow(z, 2)))

# Source: https://arxiv.org/pdf/1902.02603.pdf
def ive_fraction_approx2(v, z, eps=1e-20):
    """
    More accurate approximation of I_v(z) / I_{v-1}(z)
    Uses average of two bounds for better accuracy.
    """
    def delta_a(a):
        lamb = v + (a - 1.0) / 2.0
        return (v - 0.5) + lamb / (
            2 * torch.sqrt(torch.clamp(torch.pow(lamb, 2) + torch.pow(z, 2), min=eps))
        )

    delta_0 = delta_a(0.0)
    delta_2 = delta_a(2.0)
    B_0 = z / (delta_0 + torch.sqrt(torch.clamp(torch.pow(delta_0, 2) + torch.pow(z, 2), min=eps)))
    B_2 = z / (delta_2 + torch.sqrt(torch.clamp(torch.pow(delta_2, 2) + torch.pow(z, 2), min=eps)))

    return (B_0 + B_2) / 2.0


# von Mises-Fisher Distribution
class VonMisesFisher(torch.distributions.Distribution):
    arg_constraints = {
        "loc": torch.distributions.constraints.real,
        "scale": torch.distributions.constraints.positive,
    }
    support = torch.distributions.constraints.real
    has_rsample = True
    _mean_carrier_measure = 0

    def __init__(self, loc, scale, validate_args=None, k=1):
        self.dtype = loc.dtype
        self.loc = loc
        self.scale = scale
        self.device = loc.device
        self.__m = loc.shape[-1]
        self.__e1 = torch.Tensor([1.0] + [0] * (loc.shape[-1] - 1)).to(self.device)
        self.k = k

        super().__init__(self.loc.size(), validate_args=validate_args)

    @property
    def mean(self):
        # Use paper-based approximation for numerical stability
        return self.loc * ive_fraction_approx2(self.__m / 2, self.scale)

    @property
    def stddev(self):
        return self.scale

    def sample(self, shape=torch.Size()):
        with torch.no_grad():
            return self.rsample(shape)

    def rsample(self, shape=torch.Size()):
        shape = shape if isinstance(shape, torch.Size) else torch.Size([shape])

        w = (
            self.__sample_w3(shape=shape)
            if self.__m == 3
            else self.__sample_w_rej(shape=shape)
        )

        v = (
            torch.distributions.Normal(0, 1)
            .sample(shape + torch.Size(self.loc.shape))
            .to(self.device)
            .transpose(0, -1)[1:]
        ).transpose(0, -1)
        v = v / v.norm(dim=-1, keepdim=True)

        w_ = torch.sqrt(torch.clamp(1 - (w ** 2), 1e-10))
        x = torch.cat((w, w_ * v), -1)
        z = self.__householder_rotation(x)

        return z.type(self.dtype)

    def __sample_w3(self, shape):
        shape = shape + torch.Size(self.scale.shape)
        u = torch.distributions.Uniform(0, 1).sample(shape).to(self.device)
        self.__w = (
            1
            + torch.stack(
                [torch.log(u), torch.log(1 - u) - 2 * self.scale], dim=0
            ).logsumexp(0)
            / self.scale
        )
        return self.__w

    def __sample_w_rej(self, shape):
        c = torch.sqrt((4 * (self.scale ** 2)) + (self.__m - 1) ** 2)
        b_true = (-2 * self.scale + c) / (self.__m - 1)

        b_app = (self.__m - 1) / (4 * self.scale)
        s = torch.min(
            torch.max(
                torch.tensor([0.0], dtype=self.dtype, device=self.device),
                self.scale - 10,
            ),
            torch.tensor([1.0], dtype=self.dtype, device=self.device),
        )
        b = b_app * s + b_true * (1 - s)

        a = (self.__m - 1 + 2 * self.scale + c) / 4
        d = (4 * a * b) / (1 + b) - (self.__m - 1) * math.log(self.__m - 1)

        self.__b, (self.__e, self.__w) = b, self.__while_loop(b, a, d, shape, k=self.k)
        return self.__w

    @staticmethod
    def first_nonzero(x, dim, invalid_val=-1):
        mask = x > 0
        idx = torch.where(
            mask.any(dim=dim),
            mask.float().argmax(dim=1).squeeze(),
            torch.tensor(invalid_val, device=x.device),
        )
        return idx

    def __while_loop(self, b, a, d, shape, k=20, eps=1e-20):
        b, a, d = [
            e.repeat(*shape, *([1] * len(self.scale.shape))).reshape(-1, 1)
            for e in (b, a, d)
        ]
        w, e, bool_mask = (
            torch.zeros_like(b).to(self.device),
            torch.zeros_like(b).to(self.device),
            (torch.ones_like(b) == 1).to(self.device),
        )

        sample_shape = torch.Size([b.shape[0], k])
        shape = shape + torch.Size(self.scale.shape)

        while bool_mask.sum() != 0:
            con1 = torch.tensor((self.__m - 1) / 2, dtype=torch.float64)
            con2 = torch.tensor((self.__m - 1) / 2, dtype=torch.float64)
            e_ = (
                torch.distributions.Beta(con1, con2)
                .sample(sample_shape)
                .to(self.device)
                .type(self.dtype)
            )

            u = (
                torch.distributions.Uniform(0 + eps, 1 - eps)
                .sample(sample_shape)
                .to(self.device)
                .type(self.dtype)
            )

            w_ = (1 - (1 + b) * e_) / (1 - (1 - b) * e_)
            t = (2 * a * b) / (1 - (1 - b) * e_)

            accept = ((self.__m - 1.0) * t.log() - t + d) > torch.log(u)
            accept_idx = self.first_nonzero(accept, dim=-1, invalid_val=-1).unsqueeze(1)
            accept_idx_clamped = accept_idx.clamp(0)

            w_ = w_.gather(1, accept_idx_clamped.view(-1, 1))
            e_ = e_.gather(1, accept_idx_clamped.view(-1, 1))

            reject = accept_idx < 0
            accept = ~reject

            w[bool_mask * accept] = w_[bool_mask * accept]
            e[bool_mask * accept] = e_[bool_mask * accept]

            bool_mask[bool_mask * accept] = reject[bool_mask * accept]

        return e.reshape(shape), w.reshape(shape)

    def __householder_rotation(self, x):
        u = self.__e1 - self.loc
        u = u / (u.norm(dim=-1, keepdim=True) + 1e-5)
        z = x - 2 * (x * u).sum(-1, keepdim=True) * u
        return z

    def entropy(self):
        # Use paper-based approximation: -scale * (I_m/2 / I_{m/2-1}) + log_norm
        output = -self.scale * ive_fraction_approx2(self.__m / 2, self.scale)
        return output.view(*(output.shape[:-1])) + self._log_normalization()

    def log_prob(self, x):
        return self._log_unnormalized_prob(x) - self._log_normalization()

    def _log_unnormalized_prob(self, x):
        output = self.scale * (self.loc * x).sum(-1, keepdim=True)
        return output.view(*(output.shape[:-1]))

    def _log_normalization(self):
        # Use stable log computation
        # log C_m(kappa) = (m/2 - 1) * log(kappa) - (m/2) * log(2*pi) - log(I_{m/2-1}(kappa))
        # where log(I_v(kappa)) = log(ive(v, kappa)) + kappa
        output = -(
            (self.__m / 2 - 1) * torch.log(self.scale)
            - (self.__m / 2) * math.log(2 * math.pi)
            - (self.scale + torch.log(ive(self.__m / 2 - 1, self.scale) + 1e-30))
        )
        return output.view(*(output.shape[:-1]))


# KL divergence: KL(vMF || Uniform on sphere)
def kl_vmf_uniform(vmf, hyu):
    return -vmf.entropy() + hyu.entropy()

## 3. Dataset and DataLoader

In [ ]:
class SemanticWatermarkDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = os.path.join(root_dir, split)
        self.transform = transform
        self.categories = ['normal', 'violence', 'sexual']
        self.label_to_idx = {cat: idx for idx, cat in enumerate(self.categories)}

        self.image_paths = []
        self.labels = []

        for cat in self.categories:
            cat_dir = os.path.join(self.root_dir, cat)
            if not os.path.exists(cat_dir):
                print(f"Warning: Directory {cat_dir} does not exist")
                continue
            for img_name in os.listdir(cat_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(cat_dir, img_name))
                    self.labels.append(self.label_to_idx[cat])

        print(f"{split} dataset: {len(self.image_paths)} images")
        for cat in self.categories:
            count = sum(1 for label in self.labels if label == self.label_to_idx[cat])
            print(f"  {cat}: {count}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img_path = self.image_paths[idx]
            image = Image.open(img_path).convert('RGB')
            label = self.labels[idx]

            if self.transform:
                image = self.transform(image)

            return image, label
        except Exception as e:
            print(f"Warning: Error loading image {self.image_paths[idx]}: {e}")
            # Try next image (circularly) to avoid failure
            return self.__getitem__((idx + 1) % len(self))

In [ ]:
# ============================================================================
# DATASET PATHS CONFIGURATION
# Colab: /content/semantic_wm/dataset (auto-created by the unzip cell above)
# Local: ./dataset (after running preprocessing/prepare_dataset.py from repo root)
# ============================================================================
import sys
IN_COLAB = 'google.colab' in sys.modules
dataset_root = '/content/semantic_wm/dataset' if IN_COLAB else './dataset'

if not os.path.exists(dataset_root):
    print(f"⚠  Dataset path not found: {dataset_root}")
    print("    Colab: place semantic_wm/dataset.zip on Drive and re-run the unzip cell.")
    print("    Local: run preprocessing/prepare_dataset.py first.")
else:
    print(f"✓ Dataset path found: {dataset_root}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

try:
    train_dataset = SemanticWatermarkDataset(dataset_root, split='train', transform=transform)
    test_dataset = SemanticWatermarkDataset(dataset_root, split='test', transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
except Exception as e:
    print(f"❌ Error loading dataset: {e}")

## 4. CLIP Embedding Extraction

In [ ]:
# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

print(f"CLIP config:")
print(f"  Vision hidden size: {clip_model.config.vision_config.hidden_size}")
print(f"  Projection dim: {clip_model.config.projection_dim}")

def get_clip_embeddings(images):
    """Extract CLIP embeddings (512D) from images"""
    with torch.no_grad():
        # Process images
        inputs = clip_processor(images=images, return_tensors="pt", padding=True)
        # Move inputs to device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Method 1: Use vision model + projection manually
        vision_outputs = clip_model.vision_model(**inputs)
        image_embeds = vision_outputs[1]  # pooled_output (batch_size, hidden_size=768)

        # Apply visual projection to get 512D embeddings
        image_features = clip_model.visual_projection(image_embeds)  # (batch_size, 512)

        # Normalize embeddings (L2 normalization with epsilon for stability)
        image_features = F.normalize(image_features, p=2, dim=-1, eps=1e-8)

        # Check for NaN
        if torch.isnan(image_features).any():
            print("Warning: NaN detected in CLIP embeddings")
            image_features = torch.nan_to_num(image_features, nan=0.0)

    return image_features

## 5. Hyperspherical VAE Model

In [ ]:
class HypersphericalVAE(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, latent_dim=100):
        """
        Hyperspherical VAE with von Mises-Fisher distribution

        Args:
            input_dim: CLIP embedding dimension (512)
            hidden_dim: Hidden layer dimension
            latent_dim: Latent dimension (will be on S^(latent_dim-1) sphere)
        """
        super(HypersphericalVAE, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim

        # Encoder
        self.fc_e0 = nn.Linear(input_dim, hidden_dim * 2)
        self.fc_e1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_mean = nn.Linear(hidden_dim, latent_dim)  # Direction μ
        self.fc_kappa = nn.Linear(hidden_dim, 1)  # Concentration κ

        # Decoder
        self.fc_d0 = nn.Linear(latent_dim, hidden_dim)
        self.fc_d1 = nn.Linear(hidden_dim, hidden_dim * 2)
        self.fc_out = nn.Linear(hidden_dim * 2, input_dim)

    def encode(self, x):
        """Encode input to vMF distribution parameters"""
        h = F.relu(self.fc_e0(x))
        h = F.relu(self.fc_e1(h))

        # Mean direction on hypersphere (normalized with epsilon for stability)
        z_mean = self.fc_mean(h)
        z_mean_norm = z_mean.norm(dim=-1, keepdim=True)
        z_mean = z_mean / (z_mean_norm + 1e-8)  # Add epsilon to prevent division by zero

        # Concentration parameter (must be positive, add 1 to prevent collapse)
        z_kappa = F.softplus(self.fc_kappa(h)) + 1

        return z_mean, z_kappa

    def decode(self, z):
        """Decode latent to reconstruction (normalized to unit sphere)"""
        h = F.relu(self.fc_d0(z))
        h = F.relu(self.fc_d1(h))
        recon = self.fc_out(h)
        # Normalize to unit sphere (CLIP embeddings are L2 normalized)
        recon = F.normalize(recon, p=2, dim=-1)
        return recon

    def reparameterize(self, z_mean, z_kappa):
        """Sample from vMF distribution"""
        q_z = VonMisesFisher(z_mean, z_kappa)
        p_z = HypersphericalUniform(self.latent_dim - 1, device=z_mean.device)
        return q_z, p_z

    def forward(self, x):
        # Encode
        z_mean, z_kappa = self.encode(x)

        # Reparameterization
        q_z, p_z = self.reparameterize(z_mean, z_kappa)
        z = q_z.rsample()

        # Decode
        recon = self.decode(z)

        return recon, z, z_mean, z_kappa, q_z, p_z

## 6. Loss Function

In [ ]:
def hyperspherical_vae_loss(recon, target, q_z, p_z, beta=0.01):
    """
    Hyperspherical VAE Loss = Reconstruction Loss + β * KL(vMF || Uniform)

    Args:
        recon: Reconstructed embeddings (L2 normalized)
        target: Target CLIP embeddings (L2 normalized)
        q_z: Posterior vMF distribution
        p_z: Prior uniform distribution on sphere
        beta: KL weight
    """
    # Reconstruction loss using cosine similarity
    # cosine_similarity returns values in [-1, 1], where 1 means identical
    # We want to minimize, so use 1 - cosine_similarity
    cosine_sim = F.cosine_similarity(recon, target, dim=-1)
    recon_loss = (1 - cosine_sim).mean()

    # KL divergence: KL(vMF || Uniform on sphere)
    kl_loss = kl_vmf_uniform(q_z, p_z).mean()

    # Total loss
    total_loss = recon_loss + beta * kl_loss

    return total_loss, recon_loss, kl_loss

## 7. Training Function

In [ ]:
def train_epoch(model, clip_model, train_loader, optimizer, device, beta=0.01):
    model.train()
    train_losses = defaultdict(list)

    for images, labels in tqdm(train_loader, desc='Training'):
        images = images.to(device)

        # Extract CLIP embeddings
        with torch.no_grad():
            clip_embeddings = get_clip_embeddings([transforms.ToPILImage()(img.cpu()) for img in images])

        # Forward pass
        recon, z, z_mean, z_kappa, q_z, p_z = model(clip_embeddings)

        # Compute loss
        loss, recon_loss, kl_loss = hyperspherical_vae_loss(recon, clip_embeddings, q_z, p_z, beta)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping to prevent instability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Record losses
        train_losses['total'].append(loss.item())
        train_losses['recon'].append(recon_loss.item())
        train_losses['kl'].append(kl_loss.item())

    return {k: np.mean(v) for k, v in train_losses.items()}

def validate(model, clip_model, test_loader, device, beta=0.01):
    model.eval()
    val_losses = defaultdict(list)
    all_z = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Validation'):
            images = images.to(device)

            # Extract CLIP embeddings
            clip_embeddings = get_clip_embeddings([transforms.ToPILImage()(img.cpu()) for img in images])

            # Forward pass
            recon, z, z_mean, z_kappa, q_z, p_z = model(clip_embeddings)

            # Compute loss
            loss, recon_loss, kl_loss = hyperspherical_vae_loss(recon, clip_embeddings, q_z, p_z, beta)

            # Record losses
            val_losses['total'].append(loss.item())
            val_losses['recon'].append(recon_loss.item())
            val_losses['kl'].append(kl_loss.item())

            # Store latent representations
            all_z.append(z.cpu())
            all_labels.extend(labels.cpu().numpy())

    all_z = torch.cat(all_z, dim=0).numpy()
    all_labels = np.array(all_labels)

    return {k: np.mean(v) for k, v in val_losses.items()}, all_z, all_labels

## 8. Visualization Functions

In [ ]:
def plot_loss_curves(train_history, val_history):
    """Plot training and validation loss curves"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    loss_types = ['total', 'recon', 'kl']
    titles = ['Total Loss', 'Reconstruction Loss', 'KL Divergence']

    for ax, loss_type, title in zip(axes, loss_types, titles):
        ax.plot(train_history[loss_type], label='Train', marker='o')
        ax.plot(val_history[loss_type], label='Val', marker='s')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def verify_sphere_constraint(z):
    """Verify that latent points lie on unit hypersphere"""
    norms = np.linalg.norm(z, axis=1)

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.hist(norms, bins=50, edgecolor='black', alpha=0.7)
    plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Unit norm')
    plt.xlabel('||z||')
    plt.ylabel('Frequency')
    plt.title('Distribution of Latent Norms')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.boxplot(norms, vert=True)
    plt.axhline(1.0, color='red', linestyle='--', linewidth=2, label='Unit norm')
    plt.ylabel('||z||')
    plt.title('Latent Norm Statistics')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"Mean norm: {norms.mean():.6f}")
    print(f"Std norm: {norms.std():.6f}")
    print(f"Min norm: {norms.min():.6f}")
    print(f"Max norm: {norms.max():.6f}")
    print(f"Deviation from unit sphere: {np.abs(norms - 1.0).mean():.6f}")

def plot_tsne(z, labels, categories=['normal', 'violence', 'sexual']):
    """Plot t-SNE visualization of latent space"""
    print("Computing t-SNE...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    z_tsne = tsne.fit_transform(z)

    plt.figure(figsize=(10, 8))
    colors = ['blue', 'red', 'green']

    for i, (cat, color) in enumerate(zip(categories, colors)):
        mask = labels == i
        plt.scatter(z_tsne[mask, 0], z_tsne[mask, 1],
                   c=color, label=cat, alpha=0.6, s=30)

    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.title('t-SNE Visualization of Hyperspherical Latent Space')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def compute_cluster_metrics(z, labels):
    """Compute clustering quality metrics"""
    if len(np.unique(labels)) < 2:
        print("Not enough classes for clustering metrics")
        return

    silhouette = silhouette_score(z, labels)
    davies_bouldin = davies_bouldin_score(z, labels)
    calinski_harabasz = calinski_harabasz_score(z, labels)

    print("\n" + "="*50)
    print("Cluster Quality Metrics")
    print("="*50)
    print(f"Silhouette Score: {silhouette:.4f}")
    print(f"  (Higher is better, range [-1, 1])")
    print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")
    print(f"  (Lower is better, minimum 0)")
    print(f"Calinski-Harabasz Index: {calinski_harabasz:.4f}")
    print(f"  (Higher is better, no fixed range)")
    print("="*50)

    return {
        'silhouette': silhouette,
        'davies_bouldin': davies_bouldin,
        'calinski_harabasz': calinski_harabasz
    }

def analyze_cluster_distances(z, labels, categories=['normal', 'violence', 'sexual']):
    """Analyze intra-class and inter-class distances"""
    from scipy.spatial.distance import cdist

    print("\n" + "="*50)
    print("Intra-class and Inter-class Distance Analysis")
    print("="*50)

    # Intra-class distances
    intra_distances = []
    for i, cat in enumerate(categories):
        mask = labels == i
        if mask.sum() > 1:
            z_class = z[mask]
            distances = cdist(z_class, z_class, metric='euclidean')
            # Take upper triangle (exclude diagonal)
            triu_indices = np.triu_indices_from(distances, k=1)
            intra_dist = distances[triu_indices].mean()
            intra_distances.append(intra_dist)
            print(f"Intra-class distance ({cat}): {intra_dist:.4f}")

    print(f"\nMean intra-class distance: {np.mean(intra_distances):.4f}")

    # Inter-class distances
    inter_distances = []
    for i in range(len(categories)):
        for j in range(i+1, len(categories)):
            mask_i = labels == i
            mask_j = labels == j
            if mask_i.sum() > 0 and mask_j.sum() > 0:
                z_i = z[mask_i]
                z_j = z[mask_j]
                distances = cdist(z_i, z_j, metric='euclidean')
                inter_dist = distances.mean()
                inter_distances.append(inter_dist)
                print(f"\nInter-class distance ({categories[i]} vs {categories[j]}): {inter_dist:.4f}")

    print(f"\nMean inter-class distance: {np.mean(inter_distances):.4f}")
    print(f"Separation ratio (inter/intra): {np.mean(inter_distances)/np.mean(intra_distances):.4f}")
    print("  (Higher ratio indicates better cluster separation)")
    print("="*50)

## 9. Training Loop

In [ ]:
# Hyperparameters
INPUT_DIM = 512  # CLIP embedding dimension
HIDDEN_DIM = 256
LATENT_DIM = 100  # Latent on S^99
BETA = 0.01  # KL weight
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20

# Initialize model
model = HypersphericalVAE(INPUT_DIM, HIDDEN_DIM, LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Latent dimension: {LATENT_DIM} (on S^{LATENT_DIM-1} hypersphere)")

In [ ]:
# Training loop
train_history = defaultdict(list)
val_history = defaultdict(list)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    # Train
    train_losses = train_epoch(model, clip_model, train_loader, optimizer, device, BETA)
    for k, v in train_losses.items():
        train_history[k].append(v)

    # Validate
    val_losses, val_z, val_labels = validate(model, clip_model, test_loader, device, BETA)
    for k, v in val_losses.items():
        val_history[k].append(v)

    # Print losses
    print(f"Train Loss: {train_losses['total']:.4f} "
          f"(Recon: {train_losses['recon']:.4f}, KL: {train_losses['kl']:.4f})")
    print(f"Val Loss: {val_losses['total']:.4f} "
          f"(Recon: {val_losses['recon']:.4f}, KL: {val_losses['kl']:.4f})")

    # Verify sphere constraint every 5 epochs
    if (epoch + 1) % 5 == 0:
        norms = np.linalg.norm(val_z, axis=1)
        print(f"Latent norm: {norms.mean():.6f} ± {norms.std():.6f}")

print("\nTraining completed!")

## 10. Results Visualization and Analysis

In [ ]:
# Plot loss curves
plot_loss_curves(train_history, val_history)

In [ ]:
# Get final test embeddings
print("Getting final test embeddings...")
_, final_z, final_labels = validate(model, clip_model, test_loader, device, BETA)